<a href="https://colab.research.google.com/github/mursyidproject/olist-ecommerce-delivery-analysis/blob/main/Olist_Delivery_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Sesuaikan path-nya jika kamu pakai Google Drive
orders = pd.read_csv('olist_orders_dataset.csv')
print("Data Orders berhasil di-load! Jumlah baris dan kolom:", orders.shape)

Data Orders berhasil di-load! Jumlah baris dan kolom: (99441, 8)


In [ ]:
import pandas as pd
import numpy as np

# 1. Load keempat dataset (jika belum)
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

# 2. Filter hanya status 'delivered'
df_orders = orders[orders['order_status'] == 'delivered'].copy()

# 3. Konversi kolom timestamp dari string ke Datetime
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

# 4. Hapus baris yang missing value pada tanggal diterima pelanggan (jika ada anomaly)
df_orders = df_orders.dropna(subset=['order_delivered_customer_date'])

# 5. Feature Engineering: Hitung keterlambatan (Hari Actual - Hari Estimasi)
# Nilai positif (+) = Terlambat (Delay)
# Nilai negatif (-) = Lebih cepat sampai dari estimasi
df_orders['delivery_delay_days'] = (
    df_orders['order_delivered_customer_date'] - df_orders['order_estimated_delivery_date']
).dt.total_seconds() / (24 * 3600)

# 6. Flag Is_Late: 1 jika terlambat (> 0 hari), 0 jika tepat waktu/lebih cepat
df_orders['is_late'] = np.where(df_orders['delivery_delay_days'] > 0, 1, 0)

# Cek hasil cleaning & kolom baru
print(f"Jumlah data delivered yang siap dianalisis: {len(df_orders)} baris")
print("\nStatistik Deskriptif Keterlambatan (Hari):")
print(df_orders['delivery_delay_days'].describe())

Jumlah data delivered yang siap dianalisis: 96470 baris

Statistik Deskriptif Keterlambatan (Hari):
count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: delivery_delay_days, dtype: float64


In [ ]:
# 1. Merge df_orders dengan customers untuk mendapatkan kolom 'customer_state'
df_merged = df_orders.merge(customers[['customer_id', 'customer_state', 'customer_city']], on='customer_id', how='left')

# 2. Agregasi Performa Logistik per State
state_performance = df_merged.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    avg_delay_all_days=('delivery_delay_days', 'mean')
).reset_index()

# 3. Hitung Late Rate (%)
state_performance['late_rate_pct'] = (state_performance['late_orders'] / state_performance['total_orders']) * 100

# 4. Hitung rata-rata hari keterlambatan KHUSUS untuk paket yang terlambat (is_late == 1)
late_only = df_merged[df_merged['is_late'] == 1].groupby('customer_state')['delivery_delay_days'].mean().reset_index()
late_only.rename(columns={'delivery_delay_days': 'avg_days_when_late'}, inplace=True)

state_performance = state_performance.merge(late_only, on='customer_state', how='left')

# 5. Tampilkan Top 5 State dengan Late Rate Tertinggi (Paling Bermasalah)
top_problem_states = state_performance.sort_values(by='late_rate_pct', ascending=False).head(5)

print("=== TOP 5 WILAYAH (STATE) DENGAN TINGKAT KETERLAMBATAN TERTINGGI ===")
print(top_problem_states[['customer_state', 'total_orders', 'late_orders', 'late_rate_pct', 'avg_days_when_late']])

=== TOP 5 WILAYAH (STATE) DENGAN TINGKAT KETERLAMBATAN TERTINGGI ===
   customer_state  total_orders  late_orders  late_rate_pct  \
1              AL           397           95      23.929471   
9              MA           717          141      19.665272   
16             PI           476           76      15.966387   
5              CE          1279          196      15.324472   
24             SE           335           51      15.223881   

    avg_days_when_late  
1             9.246130  
9             9.970709  
16           12.298219  
5            14.326787  
24           16.890978  


In [ ]:
# 1. Merge df_merged dengan data reviews
df_full = df_merged.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')

# Drop missing values jika ada order yang tidak memiliki review
df_full = df_full.dropna(subset=['review_score'])

# 2. Perbandingan Rata-rata Review Score: Tepat Waktu vs Terlambat
review_comparison = df_full.groupby('is_late')['review_score'].agg(['count', 'mean', 'median']).reset_index()
review_comparison['Status'] = review_comparison['is_late'].map({0: 'On-Time / Early', 1: 'Late Delivery'})

print("=== PERBANDINGAN REVIEW SCORE: ON-TIME VS LATE ===")
print(review_comparison[['Status', 'count', 'mean', 'median']])

# 3. Distribusi Rating Bintang (1-5) berdasarkan Status Pengiriman (%)
rating_dist = pd.crosstab(df_full['is_late'], df_full['review_score'], normalize='index') * 100
rating_dist.index = ['On-Time / Early', 'Late Delivery']

print("\n=== DISTRIBUSI RATING BINTANG (%) ===")
print(rating_dist.round(2))

# 4. Hitung Korelasi Pearson antara Hari Keterlambatan dan Review Score
# Hanya dihitung pada subset paket yang memang TERLAMBAT agar polanya jernih
late_df = df_full[df_full['is_late'] == 1]
correlation = late_df['delivery_delay_days'].corr(late_df['review_score'])

print(f"\nKoefisien Korelasi (Pearson) antara Hari Keterlambatan & Review Score: {correlation:.4f}")

=== PERBANDINGAN REVIEW SCORE: ON-TIME VS LATE ===
            Status  count      mean  median
0  On-Time / Early  88653  4.293718     5.0
1    Late Delivery   7700  2.566494     2.0

=== DISTRIBUSI RATING BINTANG (%) ===
review_score       1.0   2.0    3.0    4.0    5.0
On-Time / Early   6.60  2.63   7.99  20.34  62.43
Late Delivery    46.16  7.87  11.36  12.39  22.22

Koefisien Korelasi (Pearson) antara Hari Keterlambatan & Review Score: -0.2356


In [ ]:
# 1. Pilih kolom-kolom yang relevan untuk kebutuhan Dashboard Looker Studio
columns_for_dashboard = [
    'order_id', 'customer_id', 'customer_state', 'customer_city',
    'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'delivery_delay_days', 'is_late', 'review_score'
]

df_dashboard = df_full[columns_for_dashboard].copy()

# 2. Simpan ke CSV
df_dashboard.to_csv('olist_delivery_analysis_clean.csv', index=False)
print("File 'olist_delivery_analysis_clean.csv' berhasil dibuat! Silakan unduh dari menu Files di Colab.")

# Kode otomatis mengunduh di Google Colab (opsional)
from google.colab import files
files.download('olist_delivery_analysis_clean.csv')

File 'olist_delivery_analysis_clean.csv' berhasil dibuat! Silakan unduh dari menu Files di Colab.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>